# Resumo (com cortes)

## Dataset

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/housing/housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Construção do Pipeline

In [3]:
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

### Pré-processamento dos dados

In [4]:
pipeline_log = make_pipeline(
    KNNImputer(n_neighbors=10),
    # SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

pipeline_log

,steps,"[('knnimputer', ...), ('functiontransformer', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,n_neighbors,10
,weights,'uniform'
,metric,'nan_euclidean'
,copy,True
,add_indicator,False
,keep_empty_features,False


In [5]:
from sklearn.pipeline import Pipeline

pipeline_num = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    # ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

pipeline_num

,steps,"[('impute', ...), ('standardize', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,n_neighbors,10
,weights,'uniform'
,metric,'nan_euclidean'
,copy,True
,add_indicator,False
,keep_empty_features,False


In [6]:
pipeline_cat = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

pipeline_cat

,steps,"[('simpleimputer', ...), ('onehotencoder', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,categories,'auto'


In [7]:
from sklearn.preprocessing import FunctionTransformer

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]
    
def feature_names_out(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, 
                            feature_names_out=feature_names_out),
        StandardScaler())

ratio_pipeline()

,steps,"[('simpleimputer', ...), ('functiontransformer', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,func,<function col...t 0x112967c40>


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector

preprocessing = ColumnTransformer([
    ("cat", pipeline_cat, make_column_selector(dtype_include=np.object_)),
    ("log", pipeline_log, ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(),  ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
], remainder=pipeline_num)  # latitude, longitude, housing_median_age

preprocessing

,transformers,"[('cat', ...), ('log', ...), ...]"
,remainder,Pipeline(step...ardScaler())])
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None


### Pré-processamento + Modelo

In [9]:
from sklearn.tree import DecisionTreeRegressor

In [10]:
projeto_final = make_pipeline(preprocessing, DecisionTreeRegressor())

## Características e Variável Alvo (Numérica = Regressão)

In [11]:
X, y = df.drop(columns=['median_house_value']), df['median_house_value']

## Holdout Method (Treinamento e Teste)

In [12]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2)

## Treino do Modelo de Regressão (Dataset de Treinamento)

In [13]:
projeto_final.fit(X_treino, y_treino)

,steps,"[('columntransformer', ...), ('decisiontreeregressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('log', ...), ...]"
,remainder,Pipeline(step...ardScaler())])
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Predição do Modelo de Regressão (Dataset de Teste)

In [14]:
y_predito_projeto_final = projeto_final.predict(X_teste)

## Medida de Desempenho do Modelo de Regressão

$$RMSE = \sqrt{
\frac{1}{m}\displaystyle\sum_{i=1}^{m} (y_i - \hat{y_{i}})^2}$$

In [15]:
from sklearn.metrics import root_mean_squared_error

projeto_final_rmse = root_mean_squared_error(y_teste, y_predito_projeto_final)
projeto_final_rmse

71003.33620898699